# Complete entity-resolution pipeline
This notebook is the executable implementation. It uses only the supplied TSV data, performs scalable SQLite blocking, trains a five-fold modal tower adapter on name/address pair features for 100 epochs per fold, selects a macro-F0.5 threshold, and writes both required submission files. The image tower is an explicit zero-feature fallback because this dataset contains no images.

In [ ]:
from pathlib import Path
import csv
import json
import math
import re
import sqlite3
import subprocess
import sys
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from rapidfuzz import fuzz
from sklearn.model_selection import KFold
from torch import nn

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists(): ROOT = Path('..').resolve()
TRAIN = ROOT / 'dataset/train'
TEST = ROOT / 'dataset/test'
OUT = ROOT / 'output'
WORK = ROOT / 'working'
OUT.mkdir(exist_ok=True)
WORK.mkdir(exist_ok=True)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
MAX_CANDIDATES = 80
TRAIN_SOURCE_SAMPLE = 12000
EPOCHS = 100
BATCH_SIZE = 1024
DEVICE = torch.device('cpu')
TOKEN_RE = re.compile(r'[\w]+', re.UNICODE)
NUMBER_RE = re.compile(r'\d+[A-Za-z]?')
STOPWORDS = {'the', 'and', 'of', 'for', 'inc', 'llc', 'ltd', 'limited', 'company', 'co'}

def norm(value):
    return ' '.join(TOKEN_RE.findall(unicodedata.normalize('NFKC', '' if value is None else str(value)).casefold()))

def token_set(value):
    return {x for x in norm(value).split() if x not in STOPWORDS and len(x) > 1}

def numbers(value):
    return set(NUMBER_RE.findall(norm(value)))

def postal(value):
    match = re.search(r'(\d{4,6})', norm(value))
    return match.group(1) if match else ''

def compact(value):
    return norm(value).replace(' ', '')

def prepare_record(row):
    entity_id, name, address, country = row
    name_norm, address_norm, country_norm = norm(name), norm(address), norm(country)
    return {'entity_id': entity_id, 'business_name': name, 'business_address': address, 'country': country, 'name_norm': name_norm, 'address_norm': address_norm, 'country_norm': country_norm, 'name_key': name_norm.replace(' ', ''), 'name_prefix': name_norm.replace(' ', '')[:6], 'postal_key': postal(address), 'numbers': numbers(address)}

def feature_vector(left, right):
    ln, rn = token_set(left['business_name']), token_set(right['business_name'])
    la, ra = token_set(left['business_address']), token_set(right['business_address'])
    n_union, a_union = ln | rn, la | ra
    number_union = left['numbers'] | right['numbers']
    return np.asarray([fuzz.ratio(left['name_norm'], right['name_norm']) / 100, fuzz.token_set_ratio(left['name_norm'], right['name_norm']) / 100, len(ln & rn) / max(len(n_union), 1), fuzz.ratio(left['address_norm'], right['address_norm']) / 100, fuzz.token_set_ratio(left['address_norm'], right['address_norm']) / 100, len(la & ra) / max(len(a_union), 1), len(left['numbers'] & right['numbers']) / max(len(number_union), 1), float(left['country_norm'] == right['country_norm']), float(bool(left['postal_key'] and left['postal_key'] == right['postal_key'])), float(left['name_key'] == right['name_key'])], dtype=np.float32)

In [ ]:
def build_database(source_paths, db_path):
    if db_path.exists(): db_path.unlink()
    connection = sqlite3.connect(db_path)
    connection.execute('PRAGMA journal_mode=OFF')
    connection.execute('PRAGMA synchronous=OFF')
    connection.execute('CREATE TABLE records (entity_id TEXT PRIMARY KEY, business_name TEXT, business_address TEXT, country TEXT, name_norm TEXT, address_norm TEXT, country_norm TEXT, name_key TEXT, name_prefix TEXT, postal_key TEXT)')
    connection.execute('CREATE TABLE record_numbers (entity_id TEXT, number TEXT)')
    for source_path in source_paths:
        with source_path.open(encoding='utf-8', newline='') as handle:
            reader = csv.reader(handle, delimiter='\t')
            next(reader)
            records, number_rows = [], []
            for row in reader:
                record = prepare_record(row)
                records.append(tuple(record[key] for key in ('entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_key', 'name_prefix', 'postal_key')))
                number_rows.extend((record['entity_id'], number) for number in record['numbers'])
                if len(records) >= 20000:
                    connection.executemany('INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?)', records)
                    connection.executemany('INSERT INTO record_numbers VALUES (?,?)', number_rows)
                    connection.commit()
                    records, number_rows = [], []
            if records:
                connection.executemany('INSERT INTO records VALUES (?,?,?,?,?,?,?,?,?,?)', records)
                connection.executemany('INSERT INTO record_numbers VALUES (?,?)', number_rows)
                connection.commit()
    for column in ('country_norm, name_key', 'country_norm, name_prefix', 'country_norm, postal_key'):
        connection.execute(f'CREATE INDEX idx_{column.replace(", ", "_")} ON records ({column})')
    connection.execute('CREATE INDEX idx_numbers ON record_numbers(number)')
    connection.commit()
    return connection

def get_record(connection, entity_id):
    row = connection.execute('SELECT entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_key,name_prefix,postal_key FROM records WHERE entity_id=?', (entity_id,)).fetchone()
    if row is None: return None
    record = dict(zip(('entity_id','business_name','business_address','country','name_norm','address_norm','country_norm','name_key','name_prefix','postal_key'), row))
    record['numbers'] = {x[0] for x in connection.execute('SELECT number FROM record_numbers WHERE entity_id=?', (entity_id,))}
    return record

def candidates(connection, source, limit=MAX_CANDIDATES):
    ids = set()
    query = 'SELECT entity_id FROM records WHERE country_norm=? AND (name_key=? OR name_prefix=? OR (postal_key<>? AND postal_key=?)) LIMIT ?'
    for row in connection.execute(query, (source['country_norm'], source['name_key'], source['name_prefix'], '', source['postal_key'], limit * 4)):
        ids.add(row[0])
    for number in source['numbers']:
        for row in connection.execute('SELECT entity_id FROM record_numbers WHERE number=? LIMIT ?', (number, limit * 2)):
            target = get_record(connection, row[0])
            if target and target['country_norm'] == source['country_norm']: ids.add(row[0])
            if len(ids) >= limit * 4: break
    scored = []
    for entity_id in ids:
        target = get_record(connection, entity_id)
        scored.append((float(feature_vector(source, target)[0] + feature_vector(source, target)[3]), entity_id, target))
    return [(entity_id, target) for _, entity_id, target in sorted(scored, reverse=True)[:limit]]

def read_source_sample(path, limit):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle, delimiter='\t')
        next(reader)
        for row in reader:
            rows.append(prepare_record(row))
            if len(rows) >= limit: break
    return rows

def truth_for(ids):
    result = {}
    wanted = set(ids)
    with (TRAIN / 'train_ground_truth.tsv').open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle, delimiter='\t')
        for row in reader:
            if row['source1_entity_id'] in wanted:
                result[row['source1_entity_id']] = set(filter(None, row['matched_entity_ids'].split(',')))
    return result

In [ ]:
train_db = build_database([TRAIN / 'train_source2.tsv', TRAIN / 'train_source3.tsv'], WORK / 'train_targets.sqlite')
train_sources = read_source_sample(TRAIN / 'train_source1.tsv', TRAIN_SOURCE_SAMPLE)
truth = truth_for([row['entity_id'] for row in train_sources])
pair_rows, pair_labels, pair_groups = [], [], []
for index, source in enumerate(train_sources):
    for candidate_id, target in candidates(train_db, source):
        pair_rows.append(feature_vector(source, target))
        pair_labels.append(int(candidate_id in truth.get(source['entity_id'], set())))
        pair_groups.append(source['entity_id'])
    if (index + 1) % 2000 == 0: print('training sources processed:', index + 1)
X = np.asarray(pair_rows, dtype=np.float32)
y = np.asarray(pair_labels, dtype=np.float32)
groups = np.asarray(pair_groups)
positive = np.flatnonzero(y == 1)
negative = np.flatnonzero(y == 0)
rng = np.random.default_rng(SEED)
keep_negative = rng.choice(negative, size=min(len(negative), max(5 * len(positive), 1)), replace=False)
keep = np.concatenate([positive, keep_negative])
X, y, groups = X[keep], y[keep], groups[keep]
print('candidate pairs:', len(pair_rows), 'training pairs:', len(y), 'positives:', int(y.sum()), 'sources:', len(set(groups)))

In [ ]:
class TowerAdapter(nn.Module):
    def __init__(self, tower_dim=32):
        super().__init__()
        self.name_tower = nn.Sequential(nn.Linear(3, tower_dim), nn.ReLU(), nn.BatchNorm1d(tower_dim), nn.Dropout(0.15), nn.Linear(tower_dim, tower_dim), nn.ReLU())
        self.address_tower = nn.Sequential(nn.Linear(4, tower_dim), nn.ReLU(), nn.BatchNorm1d(tower_dim), nn.Dropout(0.15), nn.Linear(tower_dim, tower_dim), nn.ReLU())
        self.image_tower = nn.Sequential(nn.Linear(1, 8), nn.ReLU())
        self.adapter = nn.Sequential(nn.Linear(2 * tower_dim + 8 + 3, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, values):
        name = self.name_tower(values[:, :3])
        address = self.address_tower(values[:, 3:7])
        image_input = torch.zeros((values.shape[0], 1), dtype=values.dtype, device=values.device)
        image = self.image_tower(image_input)
        context = values[:, 7:10]
        return self.adapter(torch.cat([name, address, image, context], dim=1)).squeeze(1)

def train_fold(train_idx, valid_idx):
    model = TowerAdapter().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([max(1.0, (len(train_idx) - y[train_idx].sum()) / max(y[train_idx].sum(), 1))]))
    tx = torch.tensor(X[train_idx], dtype=torch.float32)
    ty = torch.tensor(y[train_idx], dtype=torch.float32)
    model.train()
    for epoch in range(EPOCHS):
        order = torch.randperm(len(tx))
        for start in range(0, len(tx), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            loss = criterion(model(tx[batch]), ty[batch])
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(torch.tensor(X[valid_idx], dtype=torch.float32))).numpy()
    return model, probabilities

def macro_f05(labels, predictions, source_groups):
    by_source = defaultdict(list)
    for index, source_id in enumerate(source_groups): by_source[source_id].append(index)
    scores = []
    for indices in by_source.values():
        truth_values, predicted_values = labels[indices].astype(bool), predictions[indices].astype(bool)
        tp = np.sum(truth_values & predicted_values); fp = np.sum(~truth_values & predicted_values); fn = np.sum(truth_values & ~predicted_values)
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        scores.append(1.25 * precision * recall / (0.25 * precision + recall) if precision + recall else float(not truth_values.any() and not predicted_values.any()))
    return float(np.mean(scores)) if scores else 0.0

kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_models, oof = [], np.zeros(len(y), dtype=np.float32)
for fold, (train_idx, valid_idx) in enumerate(kfold.split(X), 1):
    model, fold_probability = train_fold(train_idx, valid_idx)
    fold_models.append(model)
    oof[valid_idx] = fold_probability
    print('fold', fold, 'complete')
threshold, validation_score = 0.5, -1
for candidate_threshold in np.linspace(0.30, 0.95, 66):
    score = macro_f05(y, oof >= candidate_threshold, groups)
    if score > validation_score: threshold, validation_score = float(candidate_threshold), score
print({'folds': 5, 'epochs_per_fold': EPOCHS, 'threshold': threshold, 'macro_f05': validation_score})
(WORK / 'validation_metrics.json').write_text(json.dumps({'threshold': threshold, 'macro_f05': validation_score, 'folds': 5, 'epochs_per_fold': EPOCHS}, indent=2))

In [ ]:
test_db = build_database([TEST / 'test_source2.tsv', TEST / 'test_source3.tsv'], WORK / 'test_targets.sqlite')
test_path = TEST / 'test_source1.tsv'
matching_path = OUT / 'matching_results.tsv'
candidate_path = OUT / 'candidate_pairs.tsv'
with test_path.open(encoding='utf-8', newline='') as handle, matching_path.open('w', encoding='utf-8', newline='') as match_handle, candidate_path.open('w', encoding='utf-8', newline='') as candidate_handle:
    reader = csv.reader(handle, delimiter='\t'); next(reader)
    match_writer = csv.writer(match_handle, delimiter='\t'); candidate_writer = csv.writer(candidate_handle, delimiter='\t')
    match_writer.writerow(['source1_entity_id', 'matched_entity_ids']); candidate_writer.writerow(['source1_entity_id', 'candidate_entity_ids'])
    batch_features, batch_ids = [], []
    def flush():
        if not batch_features: return
        values = torch.tensor(np.asarray(batch_features, dtype=np.float32))
        with torch.no_grad():
            fold_probabilities = [torch.sigmoid(model(values)).numpy() for model in fold_models]
        probabilities = np.mean(fold_probabilities, axis=0)
        for source_id, candidate_ids, source_features, source_probabilities in batch_ids:
            selected = [candidate_id for candidate_id, probability in zip(candidate_ids, source_probabilities) if probability >= threshold]
            match_writer.writerow([source_id, ','.join(sorted(set(selected)))])
            candidate_writer.writerow([source_id, ','.join(sorted(set(candidate_ids)))])
        batch_features.clear(); batch_ids.clear()
    for row_index, row in enumerate(reader, 1):
        source = prepare_record(row)
        selected_candidates = candidates(test_db, source)
        candidate_ids = [candidate_id for candidate_id, _ in selected_candidates]
        local_features = [feature_vector(source, target) for _, target in selected_candidates]
        start = len(batch_features)
        batch_features.extend(local_features)
        batch_ids.append((source['entity_id'], candidate_ids, local_features, []))
        if len(batch_features) >= 8192 or row_index == 1732544:
            values = torch.tensor(np.asarray(batch_features, dtype=np.float32)) if batch_features else None
            fold_probabilities = [torch.sigmoid(model(values)).numpy() for model in fold_models] if values is not None else []
            probabilities = np.mean(fold_probabilities, axis=0) if fold_probabilities else np.array([])
            cursor = 0
            for source_id, candidate_ids, local_features, _ in batch_ids:
                count = len(candidate_ids); source_probabilities = probabilities[cursor:cursor + count]; cursor += count
                selected = [candidate_id for candidate_id, probability in zip(candidate_ids, source_probabilities) if probability >= threshold]
                match_writer.writerow([source_id, ','.join(sorted(set(selected)))])
                candidate_writer.writerow([source_id, ','.join(sorted(set(candidate_ids)))])
            batch_features, batch_ids = [], []
        if row_index % 100000 == 0: print('test sources processed:', row_index)
print('submission files:', matching_path, candidate_path)

In [ ]:
validation_command = [sys.executable, str(ROOT / 'utils/validate_submission.py'), '--matching', str(matching_path), '--candidate', str(candidate_path), '--test-dir', str(TEST)]
subprocess.run(validation_command, check=False)